# k1 — Chia đoạn (A+B+C) + chỉ mục BM25 (CPU)

**Cài đặt:** Accelerator = **None (CPU)**, Internet = On. **Add Input:** `mir2-code`, `mir2-data`
(`data/query.parquet`, `data/links_corpus_vi_en_zh.parquet`, `corpus_out/corpus/`).

**Chạy:** *Save Version → Save & Run All*. Ước tính ~1 giờ (chia đoạn ~15 phút, tách từ pyvi/jieba phần lớn thời gian).
**Đầu ra:** `chunks/{parents,children}/`, `index/bm25/parents_{vi,en,zh}/`, `results/chunk_report.md` →
tab *Output* → *New Dataset* tên **`mir2-chunks`** (dùng cho k3, k4, k5).
Khi kho đổi (vd crawl xong longchau): tạo `mir2-data` bản mới rồi chạy lại notebook này.

In [ ]:
# (1) Chuẩn bị. Chỉ sửa các biến viết HOA ngay dưới đây.
NB_VERSION = 'k1-v1 (15/10/2026)'

import glob, os, shutil, sys, zipfile
KAGGLE = os.environ.get('KAGGLE_ROOT', '/kaggle')            # chạy thử ngoài Kaggle: đặt biến môi trường KAGGLE_ROOT
CODE = f'{KAGGLE}/working/code'
shutil.rmtree(CODE, ignore_errors=True)
hits = sorted(glob.glob(f'{KAGGLE}/input/**/mirret/__init__.py', recursive=True), key=len)
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE)
else:
    zips = sorted(glob.glob(f'{KAGGLE}/input/**/mir2_code.zip', recursive=True))
    assert zips, 'Chưa thêm dataset mir2-code (Add Input → Your Datasets)'
    zipfile.ZipFile(zips[0]).extractall(CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
from mirret.kaggle_nb import setup, sh, par, locate, locate_all, report
K = setup(NB_VERSION, {}, inputs=())

In [ ]:
# (2) Chia đoạn: đoạn cha (nộp) + đoạn con (tìm) + ngữ cảnh; báo cáo phân bố độ dài, kiểm tra bất biến
sh(f'python -m mirret.chunk run --config {K.cfg} --workers {os.cpu_count()}', 'chunk')
sh(f'python -m mirret.chunk report --config {K.cfg}', 'chunk_report')

In [ ]:
# (3) BM25 trên đoạn cha, mỗi ngôn ngữ tài liệu một chỉ mục (câu hỏi vi ↔ tài liệu vi, en ↔ en, zh ↔ zh)
K.set({'bm25.workers': os.cpu_count()})
sh(f'python -m mirret.index bm25 --config {K.cfg} --level parents', 'bm25')
report('k1')